In [1]:
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import (
    Adam,
    SGD,
    RMSprop
)
import keras_tuner as kt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout
)
import keras_tuner as kt
import pandas as pd

In [2]:
train_df = pd.read_csv("train.csv")
valid_df = pd.read_csv("validation.csv")
test_df = pd.read_csv("test.csv")

In [3]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
train_df["label"] = encoder.fit_transform(train_df["disease"])
valid_df["label"] = encoder.transform(valid_df["disease"])
test_df["label"] = encoder.transform(test_df["disease"])

In [4]:
IMG_HEIGHT = 224
IMG_WIDTH = 224
NUM_CLASSES = 7
INPUT_SHAPE = (224,224,3)
BATCH_SIZE = 32

In [5]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.2),
])

In [6]:
def load_image(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(
        image,
        channels=3
    )
    image = tf.image.resize(
        image,
        (224,224)
    )
    return image
def preprocess(path, label):
    image = load_image(path)
    image = tf.cast(image, tf.float32)
    image = tf.clip_by_value(
        image,
        0,
        255
    )
    image = image / 255.0
    image = data_augmentation(image)
    return image, label

In [7]:
train_ds = tf.data.Dataset.from_tensor_slices(
    (
        train_df["path"],
        train_df["label"]
    )
)
train_ds = train_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
train_ds = train_ds.shuffle(1000)
train_ds = train_ds.batch(BATCH_SIZE)
train_ds = train_ds.prefetch(
    tf.data.AUTOTUNE
)

In [8]:
valid_ds = tf.data.Dataset.from_tensor_slices(
    (
        valid_df["path"],
        valid_df["label"]
    )
)
valid_ds = valid_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
valid_ds = valid_ds.batch(BATCH_SIZE)
valid_ds = valid_ds.prefetch(
    tf.data.AUTOTUNE
)

In [9]:
test_ds = tf.data.Dataset.from_tensor_slices(
    (
        test_df["path"],
        test_df["label"]
    )
)
test_ds = test_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
test_ds = test_ds.batch(BATCH_SIZE)
test_ds = test_ds.prefetch(
    tf.data.AUTOTUNE
)

In [10]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)
class_weights = dict(enumerate(class_weights))
print(class_weights)

{0: np.float64(4.37305053025577), 1: np.float64(2.7817460317460316), 2: np.float64(1.3022478172023035), 3: np.float64(12.36331569664903), 4: np.float64(0.21338772031292808), 5: np.float64(1.285530900421786), 6: np.float64(10.115440115440116)}


In [11]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

In [12]:
lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [13]:
deep_cnn = Sequential([
    Conv2D(
        32,
        (3,3),
        activation="relu",
        input_shape=INPUT_SHAPE
    ),
    MaxPooling2D(2,2),
    Conv2D(
        64,
        (3,3),
        activation="relu"
    ),
    MaxPooling2D(2,2),
    Conv2D(
        128,
        (3,3),
        activation="relu"
    ),
    MaxPooling2D(2,2),
    Conv2D(
        256,
        (3,3),
        activation="relu"
    ),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(
        256,
        activation="relu"
    ),
    Dense(
        NUM_CLASSES,
        activation="softmax"
    )
])

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [14]:
deep_cnn.compile(
    optimizer="SGD",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [15]:
history_deep = deep_cnn.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=10,
    class_weight=class_weights,
    callbacks=[
        early_stop,
        lr_scheduler
    ]
)

Epoch 1/10


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 117s 523ms/step - accuracy: 0.1282 - loss: 1.9441 - val_accuracy: 0.4201 - val_loss: 1.8267 - learning_rate: 0.0100
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 116s 517ms/step - accuracy: 0.3585 - loss: 1.8909 - val_accuracy: 0.5732 - val_loss: 1.2796 - learning_rate: 0.0100
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 118s 525ms/step - accuracy: 0.3969 - loss: 1.8131 - val_accuracy: 0.6285 - val_loss: 1.2703 - learning_rate: 0.0100
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 117s 522ms/step - accuracy: 0.4237 - loss: 1.7444 - val_accuracy: 0.1225 - val_loss: 2.0882 - learning_rate: 0.0100
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 115s 513ms/step - accuracy: 0.4131 - loss: 1.7182 - val_accuracy: 0.5539 - val_loss: 1.2756 - learning_rate: 0.0100
Epoch 6/10
219/220 ━━━━━━━━━━━━━━━━━━━━ 0s 480ms/step - accuracy: 0.4449 - loss: 1.6037
Epoch 6: ReduceLROnPlateau reducing learning rate to 0.004999999888241291.
220/220 ━━━━━━━━━━━━━━━━━━━━ 115s 514ms/step - accuracy: 0.4448 - lo

In [16]:
train_loss, lr_train_acc_deep = deep_cnn.evaluate(train_ds)
valid_loss, lr_valid_acc_deep = deep_cnn.evaluate(valid_ds)
test_loss, lr_test_acc_deep = deep_cnn.evaluate(test_ds)
print(lr_train_acc_deep)
print(lr_valid_acc_deep)
print(lr_test_acc_deep)

220/220 ━━━━━━━━━━━━━━━━━━━━ 37s 161ms/step - accuracy: 0.6127 - loss: 1.0431
47/47 ━━━━━━━━━━━━━━━━━━━━ 8s 165ms/step - accuracy: 0.5919 - loss: 1.0960
47/47 ━━━━━━━━━━━━━━━━━━━━ 8s 165ms/step - accuracy: 0.5975 - loss: 1.0840
0.6126961708068848
0.5918775200843811
0.5974717140197754


In [17]:
deep_results = pd.DataFrame(columns=[
    "Model",
    "Train accuracy",
    "Test accuracy",
    "Valid accuracy"
])
deep_results.loc[len(deep_results)] = [
    "deep cnn using SGD",
    lr_train_acc_deep,
    lr_test_acc_deep,
    lr_valid_acc_deep
]
deep_results

,Model,Train accuracy,Test accuracy,Valid accuracy
0,deep cnn using SGD,0.612696,0.597472,0.591878


In [18]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

In [19]:
lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [20]:
deep_cnn = Sequential([
    Conv2D(
        32,
        (3,3),
        activation="relu",
        input_shape=INPUT_SHAPE
    ),
    MaxPooling2D(2,2),
    Conv2D(
        64,
        (3,3),
        activation="relu"
    ),
    MaxPooling2D(2,2),
    Conv2D(
        128,
        (3,3),
        activation="relu"
    ),
    MaxPooling2D(2,2),
    Conv2D(
        256,
        (3,3),
        activation="relu"
    ),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(
        256,
        activation="relu"
    ),
    Dense(
        NUM_CLASSES,
        activation="softmax"
    )
])

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [21]:
deep_cnn.compile(
    optimizer="RMSprop",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [22]:
history_deep = deep_cnn.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=10,
    class_weight=class_weights,
    callbacks=[
        early_stop,
        lr_scheduler
    ]
)

Epoch 1/10


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 116s 519ms/step - accuracy: 0.3128 - loss: 1.9402 - val_accuracy: 0.5779 - val_loss: 1.3754 - learning_rate: 0.0010
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 116s 517ms/step - accuracy: 0.3820 - loss: 1.7426 - val_accuracy: 0.5253 - val_loss: 1.1891 - learning_rate: 0.0010
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 115s 514ms/step - accuracy: 0.4238 - loss: 1.5607 - val_accuracy: 0.2730 - val_loss: 1.8099 - learning_rate: 0.0010
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 115s 514ms/step - accuracy: 0.4576 - loss: 1.5283 - val_accuracy: 0.4880 - val_loss: 1.1993 - learning_rate: 0.0010
Epoch 5/10
219/220 ━━━━━━━━━━━━━━━━━━━━ 0s 491ms/step - accuracy: 0.4796 - loss: 1.4709
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
220/220 ━━━━━━━━━━━━━━━━━━━━ 117s 525ms/step - accuracy: 0.4796 - loss: 1.4706 - val_accuracy: 0.3602 - val_loss: 1.7274 - learning_rate: 0.0010
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 116s 519ms/step - accuracy: 0.5056 - l

In [23]:
train_loss, rm_train_acc_deep = deep_cnn.evaluate(train_ds)
valid_loss, rm_valid_acc_deep = deep_cnn.evaluate(valid_ds)
test_loss, rm_test_acc_deep = deep_cnn.evaluate(test_ds)
print(rm_train_acc_deep)
print(rm_valid_acc_deep)
print(rm_test_acc_deep)

220/220 ━━━━━━━━━━━━━━━━━━━━ 37s 161ms/step - accuracy: 0.6221 - loss: 0.9197
47/47 ━━━━━━━━━━━━━━━━━━━━ 8s 165ms/step - accuracy: 0.6125 - loss: 0.9687
47/47 ━━━━━━━━━━━━━━━━━━━━ 8s 178ms/step - accuracy: 0.5935 - loss: 0.9751
0.6221112608909607
0.6125166416168213
0.5934796929359436


In [24]:
deep_results.loc[len(deep_results)] = [
    "deep cnn using RMSprop",
    rm_train_acc_deep,
    rm_test_acc_deep,
    rm_valid_acc_deep
]
deep_results

,Model,Train accuracy,Test accuracy,Valid accuracy
0,deep cnn using SGD,0.612696,0.597472,0.591878
1,deep cnn using RMSprop,0.622111,0.593480,0.612517


In [25]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

In [26]:
lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [27]:
deep_cnn = Sequential([
    Conv2D(
        32,
        (3,3),
        activation="relu",
        input_shape=INPUT_SHAPE
    ),
    MaxPooling2D(2,2),
    Conv2D(
        64,
        (3,3),
        activation="relu"
    ),
    MaxPooling2D(2,2),
    Conv2D(
        128,
        (3,3),
        activation="relu"
    ),
    MaxPooling2D(2,2),
    Conv2D(
        256,
        (3,3),
        activation="relu"
    ),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(
        256,
        activation="relu"
    ),
    Dense(
        NUM_CLASSES,
        activation="softmax"
    )
])

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [28]:
deep_cnn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [29]:
history_deep = deep_cnn.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=10,
    class_weight=class_weights,
    callbacks=[
        early_stop,
        lr_scheduler
    ]
)

Epoch 1/10


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 122s 543ms/step - accuracy: 0.3476 - loss: 1.9115 - val_accuracy: 0.5606 - val_loss: 1.5575 - learning_rate: 0.0010
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 118s 528ms/step - accuracy: 0.3857 - loss: 1.7710 - val_accuracy: 0.1385 - val_loss: 1.8150 - learning_rate: 0.0010
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 116s 519ms/step - accuracy: 0.4006 - loss: 1.7167 - val_accuracy: 0.4261 - val_loss: 1.3741 - learning_rate: 0.0010
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 116s 519ms/step - accuracy: 0.4030 - loss: 1.6258 - val_accuracy: 0.4161 - val_loss: 1.4474 - learning_rate: 0.0010
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 118s 529ms/step - accuracy: 0.3979 - loss: 1.5796 - val_accuracy: 0.2956 - val_loss: 1.9331 - learning_rate: 0.0010
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 118s 529ms/step - accuracy: 0.4512 - loss: 1.4708 - val_accuracy: 0.4820 - val_loss: 1.3353 - learning_rate: 0.0010
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 118s 528ms/step - accuracy: 0.4593 

In [30]:
train_loss, adam_train_acc_deep = deep_cnn.evaluate(train_ds)
valid_loss, adam_valid_acc_deep = deep_cnn.evaluate(valid_ds)
test_loss, adam_test_acc_deep = deep_cnn.evaluate(test_ds)
print(adam_train_acc_deep)
print(adam_valid_acc_deep)
print(adam_test_acc_deep)

220/220 ━━━━━━━━━━━━━━━━━━━━ 39s 169ms/step - accuracy: 0.5522 - loss: 1.0906
47/47 ━━━━━━━━━━━━━━━━━━━━ 8s 168ms/step - accuracy: 0.5393 - loss: 1.1295
47/47 ━━━━━━━━━━━━━━━━━━━━ 8s 167ms/step - accuracy: 0.5190 - loss: 1.1310
0.5522111058235168
0.5392809510231018
0.5189620852470398


In [31]:
deep_results.loc[len(deep_results)] = [
    "deep cnn using adam",
    rm_train_acc_deep,
    rm_test_acc_deep,
    rm_valid_acc_deep
]
deep_results

,Model,Train accuracy,Test accuracy,Valid accuracy
0,deep cnn using SGD,0.612696,0.597472,0.591878
1,deep cnn using RMSprop,0.622111,0.593480,0.612517
2,deep cnn using adam,0.622111,0.593480,0.612517


In [32]:
IMG_HEIGHT = 224
IMG_WIDTH = 224
NUM_CLASSES = 7
INPUT_SHAPE = (224,224,3)
BATCH_SIZE = 16

In [33]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.2),
])

In [34]:
def load_image(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(
        image,
        channels=3
    )
    image = tf.image.resize(
        image,
        (224,224)
    )
    return image
def preprocess(path, label):
    image = load_image(path)
    image = tf.cast(image, tf.float32)
    image = tf.clip_by_value(
        image,
        0,
        255
    )
    image = image / 255.0
    image = data_augmentation(image)
    return image, label

In [35]:
train_ds = tf.data.Dataset.from_tensor_slices(
    (
        train_df["path"],
        train_df["label"]
    )
)
train_ds = train_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
train_ds = train_ds.shuffle(1000)
train_ds = train_ds.batch(BATCH_SIZE)
train_ds = train_ds.prefetch(
    tf.data.AUTOTUNE
)

In [36]:
valid_ds = tf.data.Dataset.from_tensor_slices(
    (
        valid_df["path"],
        valid_df["label"]
    )
)
valid_ds = valid_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
valid_ds = valid_ds.batch(BATCH_SIZE)
valid_ds = valid_ds.prefetch(
    tf.data.AUTOTUNE
)

In [37]:
test_ds = tf.data.Dataset.from_tensor_slices(
    (
        test_df["path"],
        test_df["label"]
    )
)
test_ds = test_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
test_ds = test_ds.batch(BATCH_SIZE)
test_ds = test_ds.prefetch(
    tf.data.AUTOTUNE
)

In [38]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)
class_weights = dict(enumerate(class_weights))
print(class_weights)

{0: np.float64(4.37305053025577), 1: np.float64(2.7817460317460316), 2: np.float64(1.3022478172023035), 3: np.float64(12.36331569664903), 4: np.float64(0.21338772031292808), 5: np.float64(1.285530900421786), 6: np.float64(10.115440115440116)}


In [39]:
deep_cnn = Sequential([
    Conv2D(
        32,
        (3,3),
        activation="relu",
        input_shape=INPUT_SHAPE
    ),
    MaxPooling2D(2,2),
    Conv2D(
        64,
        (3,3),
        activation="relu"
    ),
    MaxPooling2D(2,2),
    Conv2D(
        128,
        (3,3),
        activation="relu"
    ),
    MaxPooling2D(2,2),
    Conv2D(
        256,
        (3,3),
        activation="relu"
    ),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(
        256,
        activation="relu"
    ),
    Dense(
        NUM_CLASSES,
        activation="softmax"
    )
])

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [40]:
deep_cnn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [41]:
batch_size = 16
history_deep = deep_cnn.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=10,
    class_weight=class_weights,
    batch_size = batch_size
)

Epoch 1/10


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


439/439 ━━━━━━━━━━━━━━━━━━━━ 121s 271ms/step - accuracy: 0.1833 - loss: 1.9415 - val_accuracy: 0.0779 - val_loss: 1.9981
Epoch 2/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 119s 268ms/step - accuracy: 0.1094 - loss: 1.8770 - val_accuracy: 0.2463 - val_loss: 1.8480
Epoch 3/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 123s 276ms/step - accuracy: 0.3177 - loss: 1.7251 - val_accuracy: 0.5586 - val_loss: 1.2918
Epoch 4/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 121s 271ms/step - accuracy: 0.3947 - loss: 1.6440 - val_accuracy: 0.5126 - val_loss: 1.3734
Epoch 5/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 121s 271ms/step - accuracy: 0.4193 - loss: 1.6263 - val_accuracy: 0.5213 - val_loss: 1.4750
Epoch 6/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 120s 269ms/step - accuracy: 0.4777 - loss: 1.4854 - val_accuracy: 0.4647 - val_loss: 1.5131
Epoch 7/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 127s 285ms/step - accuracy: 0.4652 - loss: 1.4500 - val_accuracy: 0.4640 - val_loss: 1.3035
Epoch 8/10
439/439 ━━━━━━━━━━━━━━━━━━━━ 127s 284ms/step - accuracy: 0.4472 - loss: 1.41

In [42]:
train_loss, bt16_train_acc_deep = deep_cnn.evaluate(train_ds)
valid_loss, bt16_valid_acc_deep = deep_cnn.evaluate(valid_ds)
test_loss, bt16_test_acc_deep = deep_cnn.evaluate(test_ds)
print(bt16_train_acc_deep)
print(bt16_valid_acc_deep)
print(bt16_test_acc_deep)

439/439 ━━━━━━━━━━━━━━━━━━━━ 38s 82ms/step - accuracy: 0.5683 - loss: 1.1266
94/94 ━━━━━━━━━━━━━━━━━━━━ 8s 84ms/step - accuracy: 0.5499 - loss: 1.1848
94/94 ━━━━━━━━━━━━━━━━━━━━ 8s 85ms/step - accuracy: 0.5542 - loss: 1.1655
0.5683309435844421
0.5499334335327148
0.5542249083518982


In [43]:
deep_results.loc[len(deep_results)] = [
    "deep cnn batch size 16",
    bt16_train_acc_deep,
    bt16_test_acc_deep,
    bt16_valid_acc_deep
]
deep_results

,Model,Train accuracy,Test accuracy,Valid accuracy
0,deep cnn using SGD,0.612696,0.597472,0.591878
1,deep cnn using RMSprop,0.622111,0.593480,0.612517
2,deep cnn using adam,0.622111,0.593480,0.612517
3,deep cnn batch size 16,0.568331,0.554225,0.549933


In [44]:
IMG_HEIGHT = 224
IMG_WIDTH = 224
NUM_CLASSES = 7
INPUT_SHAPE = (224,224,3)
BATCH_SIZE = 64

In [45]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.2),
])

In [46]:
def load_image(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(
        image,
        channels=3
    )
    image = tf.image.resize(
        image,
        (224,224)
    )
    return image
def preprocess(path, label):
    image = load_image(path)
    image = tf.cast(image, tf.float32)
    image = tf.clip_by_value(
        image,
        0,
        255
    )
    image = image / 255.0
    image = data_augmentation(image)
    return image, label

In [47]:
train_ds = tf.data.Dataset.from_tensor_slices(
    (
        train_df["path"],
        train_df["label"]
    )
)
train_ds = train_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
train_ds = train_ds.shuffle(1000)
train_ds = train_ds.batch(BATCH_SIZE)
train_ds = train_ds.prefetch(
    tf.data.AUTOTUNE
)

In [48]:
valid_ds = tf.data.Dataset.from_tensor_slices(
    (
        valid_df["path"],
        valid_df["label"]
    )
)
valid_ds = valid_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
valid_ds = valid_ds.batch(BATCH_SIZE)
valid_ds = valid_ds.prefetch(
    tf.data.AUTOTUNE
)

In [49]:
test_ds = tf.data.Dataset.from_tensor_slices(
    (
        test_df["path"],
        test_df["label"]
    )
)
test_ds = test_ds.map(
    preprocess,
    num_parallel_calls=tf.data.AUTOTUNE
)
test_ds = test_ds.batch(BATCH_SIZE)
test_ds = test_ds.prefetch(
    tf.data.AUTOTUNE
)

In [50]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)
class_weights = dict(enumerate(class_weights))
print(class_weights)

{0: np.float64(4.37305053025577), 1: np.float64(2.7817460317460316), 2: np.float64(1.3022478172023035), 3: np.float64(12.36331569664903), 4: np.float64(0.21338772031292808), 5: np.float64(1.285530900421786), 6: np.float64(10.115440115440116)}


In [51]:
deep_cnn = Sequential([
    Conv2D(
        32,
        (3,3),
        activation="relu",
        input_shape=INPUT_SHAPE
    ),
    MaxPooling2D(2,2),
    Conv2D(
        64,
        (3,3),
        activation="relu"
    ),
    MaxPooling2D(2,2),
    Conv2D(
        128,
        (3,3),
        activation="relu"
    ),
    MaxPooling2D(2,2),
    Conv2D(
        256,
        (3,3),
        activation="relu"
    ),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(
        256,
        activation="relu"
    ),
    Dense(
        NUM_CLASSES,
        activation="softmax"
    )
])

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [52]:
deep_cnn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [53]:
batch_size = 64
history_deep = deep_cnn.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=10,
    class_weight=class_weights,
    batch_size = batch_size
)

Epoch 1/10


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


110/110 ━━━━━━━━━━━━━━━━━━━━ 121s 1s/step - accuracy: 0.2294 - loss: 1.9667 - val_accuracy: 0.2130 - val_loss: 1.8907
Epoch 2/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 117s 1s/step - accuracy: 0.3692 - loss: 1.7865 - val_accuracy: 0.1984 - val_loss: 1.8232
Epoch 3/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 116s 1s/step - accuracy: 0.2776 - loss: 1.8028 - val_accuracy: 0.2696 - val_loss: 1.5588
Epoch 4/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 115s 1s/step - accuracy: 0.4024 - loss: 1.6875 - val_accuracy: 0.5340 - val_loss: 1.3846
Epoch 5/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 116s 1s/step - accuracy: 0.4096 - loss: 1.6108 - val_accuracy: 0.4694 - val_loss: 1.4424
Epoch 6/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 130s 1s/step - accuracy: 0.4414 - loss: 1.4692 - val_accuracy: 0.3409 - val_loss: 1.5654
Epoch 7/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 120s 1s/step - accuracy: 0.4817 - loss: 1.4326 - val_accuracy: 0.5000 - val_loss: 1.3446
Epoch 8/10
110/110 ━━━━━━━━━━━━━━━━━━━━ 117s 1s/step - accuracy: 0.4933 - loss: 1.4022 - val_accuracy: 0.434

In [54]:
train_loss, bt64_train_acc_deep = deep_cnn.evaluate(train_ds)
valid_loss, bt64_valid_acc_deep = deep_cnn.evaluate(valid_ds)
test_loss, bt64_test_acc_deep = deep_cnn.evaluate(test_ds)
print(bt64_train_acc_deep)
print(bt64_valid_acc_deep)
print(bt64_test_acc_deep)

110/110 ━━━━━━━━━━━━━━━━━━━━ 38s 333ms/step - accuracy: 0.5786 - loss: 1.0146
24/24 ━━━━━━━━━━━━━━━━━━━━ 8s 315ms/step - accuracy: 0.5533 - loss: 1.0701
24/24 ━━━━━━━━━━━━━━━━━━━━ 8s 314ms/step - accuracy: 0.5595 - loss: 1.0735
0.5786020159721375
0.5532622933387756
0.5595475435256958


In [55]:
deep_results.loc[len(deep_results)] = [
    "deep cnn batch size 64",
    bt64_train_acc_deep,
    bt64_test_acc_deep,
    bt64_valid_acc_deep
]
deep_results

,Model,Train accuracy,Test accuracy,Valid accuracy
0,deep cnn using SGD,0.612696,0.597472,0.591878
1,deep cnn using RMSprop,0.622111,0.593480,0.612517
2,deep cnn using adam,0.622111,0.593480,0.612517
3,deep cnn batch size 16,0.568331,0.554225,0.549933
4,deep cnn batch size 64,0.578602,0.559548,0.553262


In [58]:
def build_deep_cnn(hp):
    model=tf.keras.Sequential([
        Conv2D(
            hp.Int("filters1",32,64,32),
            (3,3),
            activation="relu",
            input_shape=INPUT_SHAPE
        ),
        MaxPooling2D(2,2),
        Conv2D(
            hp.Int("filters2",64,128,64),
            (3,3),
            activation="relu"
        ),
        MaxPooling2D(2,2),
        Flatten(),
        Dense(
            hp.Int(
                "units",
                64,
                256,
                64
            ),
            activation="relu"
        ),
        Dropout(
            hp.Float(
                "dropout",
                0.2,
                0.5,
                0.1
            )
        ),
        Dense(
            NUM_CLASSES,
            activation="softmax"
        )
    ])
    model.compile(
        optimizer=hp.Choice(
            "optimizer",
            ["adam","rmsprop","sgd"]
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

In [59]:
deep_tuner = kt.RandomSearch(
    build_deep_cnn,
    objective="val_accuracy",
    max_trials=5,
    directory="tuning",
    project_name="deep_cnn"
)
deep_tuner.search(
    train_ds,
    validation_data=valid_ds,
    epochs=10,
    class_weight=class_weights
)

Trial 5 Complete [00h 47m 09s]
val_accuracy: 0.5432756543159485

Best val_accuracy So Far: 0.6697736382484436
Total elapsed time: 03h 52m 55s


In [70]:
best_deep = deep_tuner.get_best_models(1)[0]

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'rm_sprop', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [71]:
best_hps = deep_tuner.get_best_hyperparameters(
    num_trials=1
)[0]
print(best_hps.values)

{'filters1': 32, 'filters2': 128, 'units': 64, 'dropout': 0.4, 'optimizer': 'rmsprop'}


In [84]:
deep_cnn = Sequential([
    Conv2D(
        32,
        (3,3),
        activation="relu",
        input_shape=INPUT_SHAPE
    ),
    MaxPooling2D(2,2),
    Conv2D(
        128,
        (3,3),
        activation="relu"
    ),
    MaxPooling2D(2,2),
    Flatten(),
    Dropout(0.4),
    Dense(
        64,
        activation="relu"
    ),
    Dense(
        NUM_CLASSES,
        activation="softmax"
    )
])

In [85]:
deep_cnn.compile(
    optimizer="RMSprop",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [86]:
history_deep = deep_cnn.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=5,
    class_weight=class_weights,
)

Epoch 1/5


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


110/110 ━━━━━━━━━━━━━━━━━━━━ 253s 2s/step - accuracy: 0.2305 - loss: 3.3026 - val_accuracy: 0.5686 - val_loss: 1.3138
Epoch 2/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 255s 2s/step - accuracy: 0.3999 - loss: 1.8148 - val_accuracy: 0.4887 - val_loss: 1.3286
Epoch 3/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 250s 2s/step - accuracy: 0.4248 - loss: 1.5892 - val_accuracy: 0.0839 - val_loss: 3.4441
Epoch 4/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 239s 2s/step - accuracy: 0.4379 - loss: 1.4909 - val_accuracy: 0.4521 - val_loss: 1.2635
Epoch 5/5
110/110 ━━━━━━━━━━━━━━━━━━━━ 246s 2s/step - accuracy: 0.4556 - loss: 1.4255 - val_accuracy: 0.4547 - val_loss: 1.3923


In [87]:
train_loss, hype_train_acc_deep = deep_cnn.evaluate(train_ds)
valid_loss, hype_valid_acc_deep = deep_cnn.evaluate(valid_ds)
test_loss, hype_test_acc_deep = deep_cnn.evaluate(test_ds)
print(hype_train_acc_deep)
print(hype_valid_acc_deep)
print(hype_test_acc_deep)

110/110 ━━━━━━━━━━━━━━━━━━━━ 57s 487ms/step - accuracy: 0.4585 - loss: 1.3461
24/24 ━━━━━━━━━━━━━━━━━━━━ 12s 489ms/step - accuracy: 0.4621 - loss: 1.3877
24/24 ━━━━━━━━━━━━━━━━━━━━ 12s 487ms/step - accuracy: 0.4538 - loss: 1.3680
0.458487868309021
0.46205058693885803
0.4537591338157654


In [75]:
deep_results.loc[len(deep_results)] = [
    "deep cnn hyperparameter",
    hype_train_acc_deep,
    hype_test_acc_deep,
    hype_valid_acc_deep
]
deep_results

,Model,Train accuracy,Test accuracy,Valid accuracy
0,deep cnn using SGD,0.612696,0.597472,0.591878
1,deep cnn using RMSprop,0.622111,0.593480,0.612517
2,deep cnn using adam,0.622111,0.593480,0.612517
3,deep cnn batch size 16,0.568331,0.554225,0.549933
4,deep cnn batch size 64,0.578602,0.559548,0.553262
5,deep cnn hyperparameter,0.576890,0.553560,0.556591


In [76]:
deep_results.to_csv("deep_cnn_comparison.csv",index=False)

In [77]:
best_deep.save("deep_cnn_phase5.keras")